# Build an Agent Bricks Knowledge Assistant on a Premium trial, wrap it with AI Gateway rate limiting plus PII guardrails, and ship a Databricks App as the UI

Eleven labs of Free Edition pattern building. One lab on Premium to put it all together at production posture. You have one session: activate the trial, ingest the corpus into the Premium workspace, stand up the Knowledge Assistant, wire AI Gateway with rate limits and PII masking, deploy a Databricks App that calls the gateway as a service principal, prove a non-admin sees less than the admin. Then tear it all down before the 14-day window closes.

> **Premium trial required.** This is the only paid lab in the cert track. The Databricks side is covered for 14 days; the underlying cloud compute is on your account (AWS, Azure, or GCP). Expect $0.50 to $5.00 in cloud spend for the 90-minute session. If you forget to tear down before day 14, the workspace converts to paid and the daily bill on a forgotten workspace can hit $50.

The hands-on work:

- Activate a Databricks Premium 14-day trial in your cloud account, deploy a Premium workspace, log in as metastore admin.
- Create catalog `labrun_genai_prod`, schema `labrun_genai_prod.default.labrun_capstone`, volume `source_docs`, re-ingest the multi-format corpus into a `chunks` table, build a `chunks_restricted` variant covering a subset, and GRANT SELECT on the restricted variant to a non-admin test user.
- Configure an Agent Bricks Knowledge Assistant named `labrun-knowledge-assistant` pointed at the chunks table. Wait for `status='READY'`.
- Create AI Gateway endpoint `labrun-gateway-endpoint` with rate limiting at 60 RPM per user, a PII masking guardrail, an inference table at `labrun_gateway_endpoint_payload`, and a usage table at `labrun_gateway_endpoint_usage`.
- Store an external MCP API key in secret scope `labrun-genai-secrets` and wire one managed MCP plus the secret-backed external MCP into the assistant's tool list.
- Write a Streamlit app that calls the gateway endpoint via the app's auto-injected service-principal credentials (NOT a user PAT, NOT a long-lived bearer token in the frontend), and deploy it as Databricks App `labrun-knowledge-app`.
- Probe the app once as the metastore admin and once as a non-admin test user. Confirm the responses differ on a question whose answer lives in a chunk the non-admin cannot SELECT. Read the inference table to see both calls attributed to distinct user identities. Read the usage table to see per-user request counts.

Estimated time: 90 minutes. Cost: $0.50 to $5.00 on your cloud account during the trial window.

> **About `critical=True`.** The brief specifies `critical=True` on the most expensive resources (the Databricks App, the gateway endpoint, the Agent Bricks app). The labrun-checks 0.7.0 `CleanupResource` shape does not yet expose a `critical` field, so this notebook OMITS the parameter to avoid a `TypeError`. Criticality is enforced semantically by manifest ordering instead (the most-expensive resources sit at the top of the manifest so they are torn down first), which matches RESOURCE-SAFETY-SPEC Rule 2.


In [ ]:
# NBVAL_SKIP
# Pinned dependencies. Streamlit is the Databricks App framework default.

!pip install --quiet labrun-checks==0.7.0 databricks-sdk==0.40.0 mlflow==2.20.0 langchain==0.3.7 databricks-langchain==0.1.1 streamlit==1.39.0


In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants for the Premium trial workspace.
# CATALOG is labrun_genai_prod (Premium does NOT pre-create `workspace`).

import atexit
import datetime
import getpass
import json
import re
import sys
import time

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "agent-bricks-knowledge-assistant-with-ai-gateway-and-databricks-app"
LAB_TAG_KEY = "labrun_lab_slug"
LAB_TAG_VALUE = LAB_ID

# Premium trial workspace: CATALOG is created by the student, not pre-existing.
CATALOG = "labrun_genai_prod"
PARENT_SCHEMA = "default"
LAB_SCHEMA = "labrun_capstone"
LAB_SCHEMA_FQN = f"{CATALOG}.{PARENT_SCHEMA}.{LAB_SCHEMA}"

VOLUME_FQN = f"{LAB_SCHEMA_FQN}.source_docs"
CHUNKS_FQN = f"{LAB_SCHEMA_FQN}.chunks"
CHUNKS_RESTRICTED_FQN = f"{LAB_SCHEMA_FQN}.chunks_restricted"
SECRET_SCOPE = "labrun-genai-secrets"
AGENT_NAME = "labrun-knowledge-assistant"
GATEWAY_ENDPOINT = "labrun-gateway-endpoint"
GATEWAY_INFERENCE_TABLE = f"{LAB_SCHEMA_FQN}.labrun_gateway_endpoint_payload"
GATEWAY_USAGE_TABLE = f"{LAB_SCHEMA_FQN}.labrun_gateway_endpoint_usage"
APP_NAME = "labrun-knowledge-app"
APP_SERVICE_PRINCIPAL = "labrun-app-sp"

# Set this to the user_name (email) of the non-admin test identity you provisioned.
# If you used a sibling service principal instead, set this to its application_id.
NON_ADMIN_USER = "non-admin-test@example.com"


In [ ]:
# NBVAL_SKIP
# Register the session and print the Premium trial cost warning + countdown.
# This lab REQUIRES an active Databricks Premium 14-day trial. The underlying
# cloud compute is billed to your cloud account, NOT to Databricks.

print("=" * 72)
print("Lab 12: Premium-trial-only. Cloud compute is on YOUR cloud bill.")
print("Expected session cost: $0.50 to $5.00 on your AWS/Azure/GCP account.")
print("If you abandon the lab without cleanup, the Databricks Premium trial")
print("converts to paid on day 15 and the daily bill on an idle workspace")
print("with a live Knowledge Assistant + gateway can hit $50/day.")
print("=" * 72)

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
databricks_host = getpass.getpass("Premium workspace URL (https://...cloud.databricks.com or .azuredatabricks.net): ").strip()
databricks_token = getpass.getpass("Databricks personal access token (PAT) for the Premium workspace: ").strip()
trial_start_input = getpass.getpass("Date you activated the Premium trial (YYYY-MM-DD): ").strip()

if not databricks_host or not databricks_token:
    print("Workspace URL and PAT are both required. Re-run this cell.")
    raise SystemExit(1)
if not databricks_host.startswith("https://"):
    databricks_host = f"https://{databricks_host}"

try:
    trial_start = datetime.date.fromisoformat(trial_start_input)
except Exception:
    print(f"Could not parse trial start date {trial_start_input!r}. Expected YYYY-MM-DD.")
    raise SystemExit(1)
trial_end = trial_start + datetime.timedelta(days=14)
days_left = (trial_end - datetime.date.today()).days
if days_left <= 0:
    print(f"WARNING: your Premium trial ended on {trial_end.isoformat()}.")
    print("The workspace has already converted to paid; cloud compute is billing")
    print("at the paid rate right now. Run the cleanup cell at the bottom of this")
    print("notebook IMMEDIATELY and tear down the workspace from the cloud console.")
elif days_left <= 3:
    print(f"WARNING: only {days_left} day(s) left before the trial converts to paid")
    print(f"(trial ends {trial_end.isoformat()}). Finish this lab and tear down today.")
else:
    print(f"Trial countdown: {days_left} day(s) remaining (trial ends {trial_end.isoformat()}).")

w = WorkspaceClient(host=databricks_host, token=databricks_token)

try:
    me = w.current_user.me()
except Exception as exc:
    print("Databricks credentials are missing or expired. CurrentUser.me() failed:")
    print(f"  {exc}")
    raise SystemExit(1)

CALLER_USER = me.user_name or (me.emails[0].value if me.emails else "unknown")
print(f"Credentials validated. Workspace user: {CALLER_USER}")

warehouses = list(w.warehouses.list())
if not warehouses:
    print("No SQL warehouses found in this Premium workspace.")
    print("Create a serverless warehouse first; the lab uses it for every UC SQL call.")
    raise SystemExit(1)
WAREHOUSE = warehouses[0]
WAREHOUSE_ID = WAREHOUSE.id
print(f"SQL warehouse resolved: {WAREHOUSE.name} ({WAREHOUSE_ID})")

DATABRICKS_CREDENTIALS = {
    "host": databricks_host,
    "token": databricks_token,
    "warehouse_id": WAREHOUSE_ID,
    "trial_end": trial_end.isoformat(),
}


def run_sql(statement, warehouse_id=None, wait_seconds=180):
    """Submit SQL to the warehouse and return {columns, rows}."""
    wid = warehouse_id or WAREHOUSE_ID
    resp = w.statement_execution.execute_statement(
        warehouse_id=wid, statement=statement, wait_timeout="50s"
    )
    statement_id = resp.statement_id
    deadline = time.time() + wait_seconds
    while True:
        state = resp.status.state if resp.status else None
        if state == StatementState.SUCCEEDED:
            break
        if state in (StatementState.FAILED, StatementState.CANCELED, StatementState.CLOSED):
            err = resp.status.error.message if (resp.status and resp.status.error) else "no error message"
            raise RuntimeError(f"SQL failed ({state}): {err}\n  Statement: {statement}")
        if time.time() > deadline:
            raise TimeoutError(
                f"SQL did not finish in {wait_seconds}s. Last state: {state}."
            )
        time.sleep(2)
        resp = w.statement_execution.get_statement(statement_id)
    columns = []
    if resp.manifest and resp.manifest.schema and resp.manifest.schema.columns:
        columns = [c.name for c in resp.manifest.schema.columns]
    rows = []
    if resp.result and resp.result.data_array:
        rows = list(resp.result.data_array)
    return {"columns": columns, "rows": rows}


register(session_token=session_token, lab_id=LAB_ID, credentials=DATABRICKS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")


In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Per RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution if any
# tagged resources from a prior session exist.
#
# Lab 12 extends the standard adapter with five new resource types:
# databricks_app, ai_gateway_inference_table, agent_bricks_app,
# databricks_secret_scope, and uc_catalog.

USER_EMAIL = CALLER_USER

CLEANUP_MANIFEST = [
    # Critical resources first (semantic ordering substitutes for the not-yet-
    # implemented critical=True flag in CleanupResource).
    CleanupResource(
        type="databricks_app",
        id=APP_NAME,
        region="databricks",
        cli_delete_command=f"databricks apps delete {APP_NAME}",
    ),
    CleanupResource(
        type="ai_gateway_inference_table",
        id=GATEWAY_USAGE_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {GATEWAY_USAGE_TABLE}\"",
    ),
    CleanupResource(
        type="ai_gateway_inference_table",
        id=GATEWAY_INFERENCE_TABLE,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {GATEWAY_INFERENCE_TABLE}\"",
    ),
    CleanupResource(
        type="model_serving_endpoint",
        id=GATEWAY_ENDPOINT,
        region="databricks",
        cli_delete_command=f"databricks serving-endpoints delete {GATEWAY_ENDPOINT}",
    ),
    CleanupResource(
        type="agent_bricks_app",
        id=AGENT_NAME,
        region="databricks",
        cli_delete_command=f"databricks agent-bricks delete {AGENT_NAME}",
    ),
    CleanupResource(
        type="databricks_secret_scope",
        id=SECRET_SCOPE,
        region="databricks",
        cli_delete_command=f"databricks secrets delete-scope --scope {SECRET_SCOPE}",
    ),
    # Standard UC resources (data plane, ordered restricted-table -> chunks ->
    # volume contents -> volume -> schema -> catalog).
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_RESTRICTED_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHUNKS_RESTRICTED_FQN}\"",
    ),
    CleanupResource(
        type="uc_managed_table",
        id=CHUNKS_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP TABLE IF EXISTS {CHUNKS_FQN}\"",
    ),
    CleanupResource(
        type="uc_volume_contents",
        id=VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks fs rm -r dbfs:/Volumes/{CATALOG}/{PARENT_SCHEMA}/{LAB_SCHEMA}/source_docs/",
    ),
    CleanupResource(
        type="uc_volume",
        id=VOLUME_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP VOLUME IF EXISTS {VOLUME_FQN}\"",
    ),
    CleanupResource(
        type="uc_schema",
        id=LAB_SCHEMA_FQN,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP SCHEMA IF EXISTS {LAB_SCHEMA_FQN} CASCADE\"",
    ),
    CleanupResource(
        type="uc_catalog",
        id=CATALOG,
        region="databricks",
        cli_delete_command=f"databricks sql -e \"DROP CATALOG IF EXISTS {CATALOG} CASCADE\"",
    ),
]


class DatabricksCleanupAdapter:
    """Inline adapter for UC + MLflow + serving + vector search resources."""

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "uc_managed_table":
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "uc_view":
            run_sql(f"DROP VIEW IF EXISTS {rid}")
        elif rtype == "uc_volume":
            run_sql(f"DROP VOLUME IF EXISTS {rid}")
        elif rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
            except Exception:
                return
            for entry in listing:
                try:
                    if entry.is_directory:
                        w.files.delete_directory(entry.path)
                    else:
                        w.files.delete(entry.path)
                except Exception:
                    pass
        elif rtype == "uc_schema":
            run_sql(f"DROP SCHEMA IF EXISTS {rid} CASCADE")
        elif rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().delete_registered_model(name=rid)
            except Exception:
                pass
        elif rtype == "mlflow_experiment":
            try:
                import mlflow
                exp = mlflow.get_experiment_by_name(rid)
                if exp is not None:
                    mlflow.delete_experiment(exp.experiment_id)
            except Exception:
                pass
        elif rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.delete(name=rid)
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.serving_endpoints.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "vector_search_index":
            try:
                w.vector_search_indexes.delete_index(index_name=rid)
            except Exception:
                pass
        elif rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.delete_endpoint(endpoint_name=rid)
            except Exception:
                pass
        else:
            raise ValueError(f"DatabricksCleanupAdapter: unknown resource type {rtype!r}")

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype in ("uc_managed_table", "uc_view"):
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, volume = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.volumes "
                    f"WHERE volume_catalog = '{catalog}' AND volume_schema = '{schema}' "
                    f"AND volume_name = '{volume}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_volume_contents":
            volume_path = "/Volumes/" + rid.replace(".", "/") + "/"
            try:
                listing = list(w.files.list_directory_contents(volume_path))
                return len(listing) > 0
            except Exception:
                return False
        if rtype == "uc_schema":
            parts = rid.split(".")
            if len(parts) != 2:
                return False
            catalog, schema = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.schemata "
                    f"WHERE catalog_name = '{catalog}' AND schema_name = '{schema}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "uc_registered_model":
            try:
                import mlflow
                mlflow.set_registry_uri("databricks-uc")
                mlflow.MlflowClient().get_registered_model(name=rid)
                return True
            except Exception:
                return False
        if rtype == "mlflow_experiment":
            try:
                import mlflow
                return mlflow.get_experiment_by_name(rid) is not None
            except Exception:
                return False
        if rtype == "model_serving_endpoint":
            try:
                w.serving_endpoints.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_index":
            try:
                w.vector_search_indexes.get_index(index_name=rid)
                return True
            except Exception:
                return False
        if rtype == "vector_search_endpoint":
            try:
                w.vector_search_endpoints.get_endpoint(endpoint_name=rid)
                return True
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        arns = []
        queries = [
            (
                "SELECT catalog_name, schema_name FROM system.information_schema.schema_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}",
            ),
            (
                "SELECT catalog_name, schema_name, table_name FROM system.information_schema.table_tags "
                f"WHERE tag_name = '{LAB_TAG_KEY}' AND tag_value = '{lab_slug}'",
                lambda row: f"{row[0]}.{row[1]}.{row[2]}",
            ),
        ]
        for sql, fmt in queries:
            try:
                result = run_sql(sql)
            except Exception:
                continue
            for row in result["rows"]:
                arns.append(fmt(row))
        return arns


class Lab12CleanupAdapter(DatabricksCleanupAdapter):
    """Lab 12 adapter: adds databricks_app, ai_gateway_inference_table,
    agent_bricks_app, databricks_secret_scope, and uc_catalog on top of the
    standard adapter shared by Labs 7-11.

    Note: the Agent Bricks Python SDK surface is still in beta as of March 2026.
    The agent_bricks_app branch tries the REST API first and raises a
    NotImplementedError with a manual-cleanup command if the API call fails.
    """

    def delete_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "databricks_app":
            try:
                w.apps.delete(name=rid)
                # Apps consume serverless DBUs while running. Wait for the
                # delete to complete before declaring success.
                deadline = time.time() + 600
                while time.time() < deadline:
                    try:
                        w.apps.get(name=rid)
                    except Exception:
                        return
                    time.sleep(5)
            except Exception:
                pass
        elif rtype == "ai_gateway_inference_table":
            # Just a UC table drop; the gateway no longer writes to it after the
            # endpoint above this entry in the manifest has been deleted.
            run_sql(f"DROP TABLE IF EXISTS {rid}")
        elif rtype == "agent_bricks_app":
            # Agent Bricks API is still in beta. Try the REST path; if it 404s
            # or the route has shifted, surface a manual-cleanup command rather
            # than silently leaking a billed resource.
            try:
                w.api_client.do("POST", f"/api/2.0/agents/{rid}/delete")
            except Exception as exc:
                raise NotImplementedError(
                    f"Could not delete Agent Bricks app {rid!r} via the REST API: {exc!r}. "
                    f"Delete it manually from the Agent Bricks UI, or run: "
                    f"`databricks agent-bricks delete {rid}` (verify the CLI command "
                    f"against the current Databricks CLI release before running)."
                )
        elif rtype == "databricks_secret_scope":
            try:
                w.secrets.delete_scope(scope=rid)
            except Exception:
                pass
        elif rtype == "uc_catalog":
            run_sql(f"DROP CATALOG IF EXISTS {rid} CASCADE")
        else:
            # Fall back to the standard adapter for uc_managed_table, uc_volume,
            # uc_volume_contents, uc_schema, model_serving_endpoint, etc.
            super().delete_resource(credentials, resource)

    def describe_resource(self, credentials, resource):
        rtype = resource.type
        rid = resource.id
        if rtype == "databricks_app":
            try:
                w.apps.get(name=rid)
                return True
            except Exception:
                return False
        if rtype == "ai_gateway_inference_table":
            parts = rid.split(".")
            if len(parts) != 3:
                return False
            catalog, schema, table = parts
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.tables "
                    f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                    f"AND table_name = '{table}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        if rtype == "agent_bricks_app":
            try:
                w.api_client.do("GET", f"/api/2.0/agents/{rid}")
                return True
            except Exception:
                return False
        if rtype == "databricks_secret_scope":
            try:
                return any(s.name == rid for s in w.secrets.list_scopes())
            except Exception:
                return False
        if rtype == "uc_catalog":
            try:
                sql = (
                    "SELECT 1 FROM system.information_schema.catalogs "
                    f"WHERE catalog_name = '{rid}'"
                )
                return len(run_sql(sql)["rows"]) > 0
            except Exception:
                return False
        return super().describe_resource(credentials, resource)


CLEANUP_ADAPTER = Lab12CleanupAdapter()


def _atexit_cleanup():
    try:
        run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans():
    return CLEANUP_ADAPTER.tag_scan(DATABRICKS_CREDENTIALS, LAB_TAG_VALUE, "databricks")


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing UC objects tagged {LAB_TAG_KEY}={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")


## Set up the Premium catalog, schema, volume, and corpus

The Premium workspace does NOT pre-create a `workspace` catalog the way Free Edition does. The first action is to create `labrun_genai_prod`, the schema `labrun_genai_prod.default.labrun_capstone`, and a UC volume for the corpus source files.

Then re-ingest the Lab 4 multi-format corpus into a `chunks` Delta table. The chunk schema is intentionally tiny (chunk_id, doc_id, content, restricted) so the lab focuses on Premium-only features instead of the ingestion mechanics covered in Lab 4. The `restricted` boolean flags which rows the non-admin user is NOT allowed to see; the lab builds a `chunks_restricted` view that filters on that flag and grants the non-admin SELECT on the view only.

After the table and view are created, GRANT `SELECT ON TABLE chunks_restricted TO <non-admin-user>` and REVOKE any existing SELECT on `chunks` for that user.


In [ ]:
# NBVAL_SKIP
# Catalog, schema, volume, chunks table + chunks_restricted view, GRANTs.

run_sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
run_sql(f"USE CATALOG {CATALOG}")
run_sql(f"CREATE SCHEMA IF NOT EXISTS {LAB_SCHEMA_FQN}")
run_sql(f"ALTER SCHEMA {LAB_SCHEMA_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
run_sql(f"CREATE VOLUME IF NOT EXISTS {VOLUME_FQN}")

# Re-ingest the Lab 4 corpus. Tiny fixture inline so the lab stands alone
# even if the student skipped the Lab 4 fixture-upload step.
CORPUS_FIXTURE = [
    (1, "doc-1", "Databricks Free Edition signs up via email OTP, Google, or Microsoft.", False),
    (2, "doc-1", "Free Edition includes one SQL warehouse and a daily compute quota.", False),
    (3, "doc-2", "Mosaic AI Vector Search Delta-Sync indexes auto-update from a Delta table.", False),
    (4, "doc-2", "Vector Search supports both storage-optimized and direct-access indexes.", False),
    (5, "doc-3", "Foundation Model API pay-per-token endpoints bill cents per session.", False),
    (6, "doc-3", "The current default chat endpoint is the Llama 3.3 70B Instruct family.", False),
    (7, "doc-4", "Unity Catalog uses three-level naming: catalog.schema.table.", False),
    (8, "doc-4", "Unity Catalog GRANTs are inherited from schema to table by default.", False),
    (9, "doc-5", "Agent Bricks Knowledge Assistant deploys a managed agent endpoint per app.", False),
    (10, "doc-5", "Agent Bricks handles retrieval, generation, and citation handling internally.", False),
    (11, "doc-6", "AI Gateway adds rate limiting, PII guardrails, inference tables, and usage tables.", False),
    (12, "doc-6", "Rate limits at 60 RPM per user require the gateway to attribute requests to user identities.", False),
    (13, "doc-7", "Databricks Apps accept Streamlit, FastAPI, Dash, Flask, and Node.js as frameworks.", False),
    (14, "doc-7", "Databricks Apps inject service-principal credentials at runtime; PATs in the frontend are forbidden.", False),
    (15, "doc-8", "MCP servers are managed (Databricks-operated), external (third-party), or custom (engineer-built).", False),
    (16, "doc-8", "External MCP servers reference API keys stored in Databricks Secret Scopes.", False),
    (17, "doc-9", "Pre-deployment evaluation runs LLM-as-judge metrics on a holdout set.", False),
    (18, "doc-9", "Post-deployment monitoring reads from the inference table written by AI Gateway.", False),
    (19, "doc-10", "Cost governance on Mosaic AI uses the usage table broken out by serving endpoint.", False),
    (20, "doc-10", "Per-user request counts in the usage table identify abusive callers.", False),
    # Restricted rows: the secret answer the non-admin user must NOT see.
    (21, "doc-secret", "The Acme Corp internal launch date for project Polaris is March 14, 2026.", True),
    (22, "doc-secret", "Project Polaris budget is $4.2M, signed off by the CFO on January 8.", True),
    (23, "doc-secret", "Project Polaris executive sponsor is the SVP of Customer Engineering.", True),
    (24, "doc-secret", "The Polaris pilot customer list contains 12 named accounts under NDA.", True),
    (25, "doc-secret", "Polaris is targeting a 99.9% uptime SLO with a four-engineer on-call rotation.", True),
    (26, "doc-public", "Acme Corp publishes a customer-facing changelog every Tuesday.", False),
    (27, "doc-public", "Acme support hours are 9 AM to 6 PM Eastern, Monday through Friday.", False),
    (28, "doc-public", "Acme offers a 30-day free trial for new customers, no credit card required.", False),
    (29, "doc-public", "The Acme documentation portal is at docs.acme.example.com.", False),
    (30, "doc-public", "Acme's security white paper is updated annually each March.", False),
]

run_sql(
    f"CREATE OR REPLACE TABLE {CHUNKS_FQN} ("
    "chunk_id INT, doc_id STRING, content STRING, restricted BOOLEAN)"
)
run_sql(f"ALTER TABLE {CHUNKS_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")

rows_literal = ",\n".join(
    f"({cid}, '{doc.replace(chr(39), chr(39)*2)}', '{content.replace(chr(39), chr(39)*2)}', {str(restricted).lower()})"
    for cid, doc, content, restricted in CORPUS_FIXTURE
)
run_sql(f"INSERT INTO {CHUNKS_FQN} VALUES {rows_literal}")
print(f"Inserted {len(CORPUS_FIXTURE)} chunks into {CHUNKS_FQN}.")

# Restricted view: only the non-restricted rows.
run_sql(
    f"CREATE OR REPLACE TABLE {CHUNKS_RESTRICTED_FQN} AS "
    f"SELECT chunk_id, doc_id, content, restricted FROM {CHUNKS_FQN} "
    "WHERE restricted = false"
)
run_sql(f"ALTER TABLE {CHUNKS_RESTRICTED_FQN} SET TAGS ('{LAB_TAG_KEY}' = '{LAB_TAG_VALUE}')")
print(f"Created {CHUNKS_RESTRICTED_FQN} (non-restricted rows only).")


## Task 1: GRANT the non-admin test user SELECT on chunks_restricted, NOT on chunks

The non-admin test user is a second identity in the workspace. The brief recommends either a second SSO user (account-admin-only) or a sibling service principal. The lab uses whichever identity name is in `NON_ADMIN_USER` at the top of the notebook.

Two GRANT statements plus the verifying SHOW GRANTS calls. The validator reads the GRANTs back via `SHOW GRANTS ON TABLE ...` and asserts the non-admin user has SELECT on `chunks_restricted` and does NOT have SELECT on `chunks`.

If your `NON_ADMIN_USER` identity does not exist in the workspace yet, the GRANT statement returns a clear UC error pointing at the missing principal. Provision the identity first (Settings -> Identity and access -> Users or Service principals -> Add).


In [ ]:
# NBVAL_SKIP
# Task 1: GRANT the non-admin user SELECT on chunks_restricted, REVOKE on chunks.

# YOUR CODE: run GRANT SELECT ON TABLE chunks_restricted TO `<NON_ADMIN_USER>`
# YOUR CODE: run REVOKE SELECT ON TABLE chunks FROM `<NON_ADMIN_USER>`
# YOUR CODE:   (this is a no-op if the user never had SELECT; the lab is
# YOUR CODE:    being explicit about what the resulting permission state is)
# YOUR CODE: run GRANT USE CATALOG ON CATALOG labrun_genai_prod TO `<NON_ADMIN_USER>`
# YOUR CODE: run GRANT USE SCHEMA ON SCHEMA labrun_genai_prod.default.labrun_capstone TO `<NON_ADMIN_USER>`
# YOUR CODE: print SHOW GRANTS ON TABLE chunks_restricted so you see the new state.


In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Premium tier metastore, chunks table has rows, chunks_restricted
# exists, non-admin user has SELECT on chunks_restricted and NOT on chunks.


def checkpoint_1(session):
    # 1. Premium-tier metastore present.
    try:
        summary = w.metastores.summary()
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Could not read metastore summary: {exc!r}. "
                f"Confirm the workspace is attached to a Premium-tier metastore."
            ),
        )
    # 2. chunks table exists with >=30 rows.
    try:
        count = int(run_sql(f"SELECT count(*) FROM {CHUNKS_FQN}")["rows"][0][0])
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Could not count rows in {CHUNKS_FQN}: {exc!r}. "
                f"Did the setup cell run and insert the corpus fixture?"
            ),
        )
    if count < 30:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{CHUNKS_FQN} has {count} rows; expected at least 30. "
                f"Re-run the setup cell."
            ),
        )
    # 3. chunks_restricted exists.
    try:
        run_sql(f"SELECT 1 FROM {CHUNKS_RESTRICTED_FQN} LIMIT 1")
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{CHUNKS_RESTRICTED_FQN} not queryable: {exc!r}. "
                f"Re-run the setup cell."
            ),
        )
    # 4. Non-admin user has SELECT on chunks_restricted.
    try:
        restricted_grants = run_sql(f"SHOW GRANTS ON TABLE {CHUNKS_RESTRICTED_FQN}")["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"Could not SHOW GRANTS on {CHUNKS_RESTRICTED_FQN}: {exc!r}",
        )
    has_select_on_restricted = any(
        NON_ADMIN_USER in str(row) and "SELECT" in str(row).upper()
        for row in restricted_grants
    )
    if not has_select_on_restricted:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Non-admin user {NON_ADMIN_USER!r} does not have SELECT on "
                f"{CHUNKS_RESTRICTED_FQN}. Run GRANT SELECT ON TABLE "
                f"{CHUNKS_RESTRICTED_FQN} TO `{NON_ADMIN_USER}`."
            ),
        )
    # 5. Non-admin user does NOT have SELECT on chunks.
    try:
        chunks_grants = run_sql(f"SHOW GRANTS ON TABLE {CHUNKS_FQN}")["rows"]
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=f"Could not SHOW GRANTS on {CHUNKS_FQN}: {exc!r}",
        )
    has_select_on_chunks = any(
        NON_ADMIN_USER in str(row) and "SELECT" in str(row).upper()
        for row in chunks_grants
    )
    if has_select_on_chunks:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Non-admin user {NON_ADMIN_USER!r} has SELECT on {CHUNKS_FQN}; "
                f"REVOKE it. Checkpoint 5 depends on the non-admin seeing only "
                f"the restricted variant."
            ),
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)


<details><summary>Hint 1 (nudge)</summary>

Two GRANT statements and a REVOKE. The validator reads grants with `SHOW GRANTS ON TABLE`. Make sure the non-admin user already exists in the workspace before running the GRANT; if the user is missing the GRANT errors with "principal not found."

</details>

<details><summary>Hint 2 (direction)</summary>

```
run_sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{NON_ADMIN_USER}`")
run_sql(f"GRANT USE SCHEMA ON SCHEMA {LAB_SCHEMA_FQN} TO `{NON_ADMIN_USER}`")
run_sql(f"GRANT SELECT ON TABLE {CHUNKS_RESTRICTED_FQN} TO `{NON_ADMIN_USER}`")
try:
    run_sql(f"REVOKE SELECT ON TABLE {CHUNKS_FQN} FROM `{NON_ADMIN_USER}`")
except Exception:
    pass  # REVOKE is a no-op if the grant never existed.
```

Then `print(run_sql(f"SHOW GRANTS ON TABLE {CHUNKS_RESTRICTED_FQN}"))` to see the state.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1:

```python
run_sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{NON_ADMIN_USER}`")
run_sql(f"GRANT USE SCHEMA ON SCHEMA {LAB_SCHEMA_FQN} TO `{NON_ADMIN_USER}`")
run_sql(f"GRANT SELECT ON TABLE {CHUNKS_RESTRICTED_FQN} TO `{NON_ADMIN_USER}`")
try:
    run_sql(f"REVOKE SELECT ON TABLE {CHUNKS_FQN} FROM `{NON_ADMIN_USER}`")
except Exception as exc:
    print(f"REVOKE noop or absent grant: {exc!r}")

grants = run_sql(f"SHOW GRANTS ON TABLE {CHUNKS_RESTRICTED_FQN}")["rows"]
print(f"Grants on {CHUNKS_RESTRICTED_FQN}:")
for row in grants:
    print(f"  {row}")
```

</details>


**Wallet check.** 6 minutes into the lab. UC metadata writes plus a 30-row insert plus four GRANT statements. Cloud-side compute is running roughly $0.20 per hour of active warehouse time; total spend so far approximately 2 cents.


## Task 2: Deploy the Agent Bricks Knowledge Assistant

Two paths: UI or SDK. The validator checks live cloud state, so either path passes.

**UI path** (recommended for first build):

1. In the Premium workspace, open the left nav and choose **Mosaic AI -> Agent Bricks -> Knowledge Assistant**.
2. Click **New Knowledge Assistant**, name it `labrun-knowledge-assistant`, and pick the lab schema `labrun_genai_prod.default.labrun_capstone` as the resource location.
3. Add the chunks table `labrun_genai_prod.default.labrun_capstone.chunks` as the knowledge source. Agent Bricks provisions a managed Vector Search index automatically.
4. Use the default backing model (the current Databricks-recommended endpoint as of March 2026, in the Llama 3.3 70B Instruct family).
5. Save and wait for the agent's `status` to transition from `CREATING` to `READY` (typically 3 to 10 minutes; Agent Bricks is provisioning the Vector Search index plus the managed serving endpoint).

**SDK path** (faster on a re-run, but the API surface is evolving):

```python
agent_payload = {
    "name": AGENT_NAME,
    "type": "knowledge_assistant",
    "knowledge_sources": [{"table_name": CHUNKS_FQN}],
    "resource_location": LAB_SCHEMA_FQN,
}
try:
    w.api_client.do("POST", "/api/2.0/agents", body=agent_payload)
except Exception as exc:
    print(f"Agent create failed (rate-evolving API): {exc!r}")
```

Either way, the validator polls the agent's status until READY with a 900-second ceiling, then issues a query against the managed agent endpoint to confirm it returns a non-empty response on a known question. Agent Bricks' default question-mode endpoint name is the agent name itself.


In [ ]:
# NBVAL_SKIP
# Task 2: deploy the Knowledge Assistant. The student picks UI or SDK above.
# This cell polls until the agent is READY, then sends one probe query.

# YOUR CODE: poll w.api_client.do('GET', f'/api/2.0/agents/{AGENT_NAME}') every 30s
# YOUR CODE:   read the status field; break when status == 'READY'
# YOUR CODE:   ceiling 900 seconds; print elapsed time
# YOUR CODE: once READY, send a probe via:
# YOUR CODE:   w.serving_endpoints.query(name=AGENT_NAME, inputs=['What is Free Edition?'])
# YOUR CODE:   wrap in try-except (the SDK shape for agent queries is evolving)
# YOUR CODE: print the response so you see the round-trip


In [ ]:
# NBVAL_SKIP
# Checkpoint 2: agent exists with status='READY', managed Vector Search index
# points at the chunks table, and a probe query returns a non-empty response.


def checkpoint_2(session):
    # 1. Agent exists.
    try:
        agent_resp = w.api_client.do("GET", f"/api/2.0/agents/{AGENT_NAME}")
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Could not GET Agent Bricks app {AGENT_NAME!r}: {exc!r}. "
                f"Confirm the agent was created via the UI or the POST /api/2.0/agents call."
            ),
        )
    status_value = None
    if isinstance(agent_resp, dict):
        status_value = agent_resp.get("status") or agent_resp.get("state")
    if status_value != "READY":
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Agent {AGENT_NAME!r} status is {status_value!r}; expected READY. "
                f"Agent Bricks provisions Vector Search and a managed endpoint; budget 3 to 10 minutes."
            ),
        )
    # 2. Probe query returns a non-empty response. Agent endpoint name == agent name.
    try:
        resp = w.serving_endpoints.query(
            name=AGENT_NAME,
            inputs=["How do I sign up for Databricks Free Edition?"],
        )
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Probe query against {AGENT_NAME!r} failed: {exc!r}. "
                f"The agent's managed endpoint may still be cold-starting; re-run after a minute."
            ),
        )
    # The response shape varies across Agent Bricks API versions; treat any
    # non-empty payload as success.
    payload = getattr(resp, "predictions", None) or getattr(resp, "choices", None)
    if not payload:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Probe query returned an empty payload. The agent is READY but the "
                f"managed endpoint did not generate a response; check the Agent Bricks "
                f"UI for index-build errors."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)


<details><summary>Hint 1 (nudge)</summary>

Agent Bricks deploys two things asynchronously: a managed Vector Search index over the chunks table, and a managed serving endpoint that wraps retrieval + generation + citation. The status field gates both. The probe query goes against the same endpoint name as the agent name.

</details>

<details><summary>Hint 2 (direction)</summary>

```
deadline = time.time() + 900
start = time.time()
while time.time() < deadline:
    info = w.api_client.do("GET", f"/api/2.0/agents/{AGENT_NAME}")
    status = info.get("status") if isinstance(info, dict) else None
    print(f"  {int(time.time() - start)}s: status={status}")
    if status == "READY":
        break
    time.sleep(30)
```

Then `w.serving_endpoints.query(name=AGENT_NAME, inputs=["..."])` for the probe.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2:

```python
deadline = time.time() + 900
start = time.time()
while time.time() < deadline:
    try:
        info = w.api_client.do("GET", f"/api/2.0/agents/{AGENT_NAME}")
    except Exception as exc:
        print(f"  {int(time.time() - start)}s: GET failed ({exc!r}); retrying")
        time.sleep(30)
        continue
    status = info.get("status") if isinstance(info, dict) else None
    print(f"  {int(time.time() - start)}s: status={status}")
    if status == "READY":
        break
    time.sleep(30)
else:
    raise TimeoutError(f"Agent {AGENT_NAME} did not reach READY in 900s")

try:
    resp = w.serving_endpoints.query(
        name=AGENT_NAME,
        inputs=["How do I sign up for Databricks Free Edition?"],
    )
    print(f"Probe response: {resp!r}")
except Exception as exc:
    print(f"Probe failed (cold-start is normal on first call): {exc!r}")
```

</details>


**Wallet check.** 22 minutes in. Agent Bricks just stood up a managed Vector Search index plus a managed serving endpoint; cloud-side compute is running roughly $0.40 per hour of active build time during provisioning; total spend so far approximately 15 cents.


## Task 3: Wire AI Gateway with rate limiting at 60 RPM, PII guardrail, inference table, usage table

Create or update model serving endpoint `labrun-gateway-endpoint`. Its `ai_gateway` config is the body that adds rate limits, the PII guardrail, the inference table, and the usage table. Its `served_entities` is a single entry that routes 100% of traffic to the Agent Bricks Knowledge Assistant's underlying entity.

**SDK path:**

```python
gateway_config = {
    "name": GATEWAY_ENDPOINT,
    "served_entities": [{
        "name": "agent-entity",
        "entity_name": AGENT_NAME,
        "entity_version": "1",
        "workload_size": "Small",
        "scale_to_zero_enabled": True,
    }],
    "ai_gateway": {
        "rate_limits": [{
            "calls": 60,
            "renewal_period": "minute",
            "key": "user",
        }],
        "guardrails": {
            "input": {"pii": {"behavior": "MASK"}},
            "output": {"pii": {"behavior": "MASK"}},
        },
        "inference_table_config": {
            "enabled": True,
            "catalog_name": CATALOG,
            "schema_name": PARENT_SCHEMA,
            "table_name_prefix": "labrun_gateway_endpoint",
        },
        "usage_tracking_config": {"enabled": True},
    },
}
```

**UI path:** Serving -> Create endpoint -> AI Gateway tab; fill rate limits, guardrails, inference table, usage tracking; on the Served Entities tab point at the Agent Bricks Knowledge Assistant.

The validator reads the endpoint's `ai_gateway` block and asserts the rate limit is 60 RPM, the guardrail is PII MASK, and both the inference and usage tables exist as Delta tables.


In [ ]:
# NBVAL_SKIP
# Task 3: create/update the AI Gateway endpoint and wait for READY.

# YOUR CODE: build the gateway_config dict shown in the markdown above
# YOUR CODE: try w.serving_endpoints.get(name=GATEWAY_ENDPOINT)
# YOUR CODE:   if it exists, update via w.serving_endpoints.update_config(...)
# YOUR CODE:   if it does not, create via w.serving_endpoints.create(...)
# YOUR CODE:   (the SDK accepts the ai_gateway block on both create and update;
# YOUR CODE:    if your SDK version does not, use w.api_client.do('PUT', ...) with
# YOUR CODE:    the JSON body)
# YOUR CODE: poll w.serving_endpoints.get(name=...) until state.ready == 'READY'
# YOUR CODE:   ceiling 900 seconds


In [ ]:
# NBVAL_SKIP
# Checkpoint 3: gateway endpoint exists with rate limit 60 RPM, PII guardrail,
# and both inference and usage tables exist as Delta tables.


def checkpoint_3(session):
    # 1. Endpoint exists.
    try:
        ep = w.serving_endpoints.get(name=GATEWAY_ENDPOINT)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Gateway endpoint {GATEWAY_ENDPOINT!r} not found: {exc!r}. "
                f"Did serving_endpoints.create() run in Task 3?"
            ),
        )
    # 2. Read the ai_gateway block (shape varies across SDK versions).
    ai_gateway = getattr(ep, "ai_gateway", None) or (
        ep.__dict__.get("ai_gateway") if hasattr(ep, "__dict__") else None
    )
    if ai_gateway is None:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Endpoint {GATEWAY_ENDPOINT!r} has no ai_gateway block. "
                f"Include the ai_gateway dict on the create or update call."
            ),
        )
    gateway_blob = json.dumps(ai_gateway, default=str).lower()
    if "60" not in gateway_blob or "minute" not in gateway_blob:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"ai_gateway rate_limits does not look like 60 calls per minute. "
                f"Observed (serialized): {gateway_blob[:300]}..."
            ),
        )
    if "pii" not in gateway_blob or ("mask" not in gateway_blob and "block" not in gateway_blob):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"ai_gateway guardrails do not include a PII MASK or BLOCK config. "
                f"Observed (serialized): {gateway_blob[:300]}..."
            ),
        )
    # 3. Inference + usage tables exist as Delta tables.
    for table_fqn in (GATEWAY_INFERENCE_TABLE, GATEWAY_USAGE_TABLE):
        try:
            catalog, schema, table = table_fqn.split(".")
            rows = run_sql(
                "SELECT 1 FROM system.information_schema.tables "
                f"WHERE table_catalog = '{catalog}' AND table_schema = '{schema}' "
                f"AND table_name = '{table}'"
            )["rows"]
        except Exception as exc:
            return CheckpointResult(
                status="fail",
                error_reason=f"Could not query information_schema for {table_fqn}: {exc!r}",
            )
        if not rows:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AI Gateway table {table_fqn} not found. The gateway writes to it on "
                    f"first request; send one probe through the gateway and wait 1 to 5 minutes."
                ),
            )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)


<details><summary>Hint 1 (nudge)</summary>

The `ai_gateway` block has four named sub-blocks: `rate_limits` (list), `guardrails` (input + output, each holding a PII config), `inference_table_config` (catalog, schema, table_name_prefix, enabled), and `usage_tracking_config` (enabled). All four must be present for the validator to pass. The inference and usage tables are materialized lazily; the first probe query through the gateway creates them.

</details>

<details><summary>Hint 2 (direction)</summary>

```
gateway_config = {
    "name": GATEWAY_ENDPOINT,
    "served_entities": [{"name": "agent-entity", "entity_name": AGENT_NAME, "workload_size": "Small", "scale_to_zero_enabled": True}],
    "ai_gateway": {
        "rate_limits": [{"calls": 60, "renewal_period": "minute", "key": "user"}],
        "guardrails": {"input": {"pii": {"behavior": "MASK"}}, "output": {"pii": {"behavior": "MASK"}}},
        "inference_table_config": {"enabled": True, "catalog_name": CATALOG, "schema_name": PARENT_SCHEMA, "table_name_prefix": "labrun_gateway_endpoint"},
        "usage_tracking_config": {"enabled": True},
    },
}

try:
    w.serving_endpoints.get(name=GATEWAY_ENDPOINT)
    w.api_client.do("PUT", f"/api/2.0/serving-endpoints/{GATEWAY_ENDPOINT}", body=gateway_config)
except Exception:
    w.api_client.do("POST", "/api/2.0/serving-endpoints", body=gateway_config)
```

Send one probe through the gateway after READY so the inference and usage tables materialize.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3:

```python
gateway_config = {
    "name": GATEWAY_ENDPOINT,
    "served_entities": [{
        "name": "agent-entity",
        "entity_name": AGENT_NAME,
        "workload_size": "Small",
        "scale_to_zero_enabled": True,
    }],
    "ai_gateway": {
        "rate_limits": [{"calls": 60, "renewal_period": "minute", "key": "user"}],
        "guardrails": {
            "input": {"pii": {"behavior": "MASK"}},
            "output": {"pii": {"behavior": "MASK"}},
        },
        "inference_table_config": {
            "enabled": True,
            "catalog_name": CATALOG,
            "schema_name": PARENT_SCHEMA,
            "table_name_prefix": "labrun_gateway_endpoint",
        },
        "usage_tracking_config": {"enabled": True},
    },
}

try:
    w.serving_endpoints.get(name=GATEWAY_ENDPOINT)
    w.api_client.do("PUT", f"/api/2.0/serving-endpoints/{GATEWAY_ENDPOINT}", body=gateway_config)
    print(f"Updated gateway endpoint {GATEWAY_ENDPOINT}")
except Exception:
    w.api_client.do("POST", "/api/2.0/serving-endpoints", body=gateway_config)
    print(f"Created gateway endpoint {GATEWAY_ENDPOINT}")

deadline = time.time() + 900
start = time.time()
while time.time() < deadline:
    ep = w.serving_endpoints.get(name=GATEWAY_ENDPOINT)
    state = getattr(ep.state, "ready", None)
    state_value = getattr(state, "value", state) if state is not None else None
    print(f"  {int(time.time() - start)}s: state={state_value}")
    if str(state_value) == "READY":
        break
    time.sleep(30)

try:
    w.serving_endpoints.query(name=GATEWAY_ENDPOINT, inputs=["warmup"])
except Exception as exc:
    print(f"Probe through gateway raised (cold-start is normal): {exc!r}")
```

</details>


**Wallet check.** 38 minutes in. The gateway endpoint provisioning is the second cold-start of the session; cloud-side compute is running roughly $0.40 per hour of active build time; total spend so far approximately 40 cents.


## Task 4: Deploy the Databricks App with service-principal auth

Three files plus a deploy call.

**`app.py`** (Streamlit handler):

```python
import os
import streamlit as st
from databricks.sdk import WorkspaceClient

# Databricks Apps inject the app's service-principal token via env vars.
# Never read a user PAT here. Never write a PAT into the JavaScript.
DATABRICKS_HOST = os.environ["DATABRICKS_HOST"]
DATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]  # auto-injected SP token

# The calling user's identity flows in the X-Forwarded-Email request header.
caller_email = st.context.headers.get("x-forwarded-email", "anonymous")

st.title("Labrun Knowledge Assistant")
question = st.text_input("Ask a question:")
if question:
    w = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)
    resp = w.serving_endpoints.query(
        name="labrun-gateway-endpoint",
        inputs=[question],
        extra_headers={"x-actor-user": caller_email},
    )
    st.write(resp.predictions[0] if resp.predictions else str(resp))
```

**`app.yaml`** (deployment manifest):

```yaml
command: ["streamlit", "run", "app.py"]
env:
  - name: DATABRICKS_HOST
    value: $DATABRICKS_HOST
  - name: DATABRICKS_TOKEN
    valueFrom: databricks-token
```

**`requirements.txt`**: `streamlit==1.39.0` and `databricks-sdk==0.40.0`.

Deploy via `databricks apps create labrun-knowledge-app` (one-time) followed by `databricks apps deploy labrun-knowledge-app --source-code-path /Workspace/Users/<you>/labrun-app`. The validator checks that `apps.get(name=APP_NAME)` returns a `RUNNING` status and that the app's config references the service principal (not a user identity) as the auth principal.


In [ ]:
# NBVAL_SKIP
# Task 4: write app source + manifest + requirements into a workspace folder,
# then create + deploy the Databricks App.

import base64

APP_SOURCE_PATH = f"/Workspace/Users/{CALLER_USER}/labrun-app"

APP_PY = '''import os\nimport streamlit as st\nfrom databricks.sdk import WorkspaceClient\n\nDATABRICKS_HOST = os.environ["DATABRICKS_HOST"]\nDATABRICKS_TOKEN = os.environ["DATABRICKS_TOKEN"]\n\ncaller_email = st.context.headers.get("x-forwarded-email", "anonymous")\n\nst.title("Labrun Knowledge Assistant")\nquestion = st.text_input("Ask a question:")\nif question:\n    w = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)\n    resp = w.serving_endpoints.query(\n        name="labrun-gateway-endpoint",\n        inputs=[question],\n        extra_headers={"x-actor-user": caller_email},\n    )\n    st.write(resp.predictions[0] if resp.predictions else str(resp))\n'''

APP_YAML = '''command: ["streamlit", "run", "app.py"]\nenv:\n  - name: DATABRICKS_HOST\n    value: $DATABRICKS_HOST\n  - name: DATABRICKS_TOKEN\n    valueFrom: databricks-token\n'''

REQUIREMENTS_TXT = '''streamlit==1.39.0\ndatabricks-sdk==0.40.0\n'''

# YOUR CODE: w.workspace.mkdirs(APP_SOURCE_PATH)
# YOUR CODE: for each of (app.py, app.yaml, requirements.txt):
# YOUR CODE:   w.workspace.upload(
# YOUR CODE:       path=f"{APP_SOURCE_PATH}/{filename}",
# YOUR CODE:       content=body.encode('utf-8'),
# YOUR CODE:       overwrite=True,
# YOUR CODE:       format='AUTO',
# YOUR CODE:   )
# YOUR CODE: try w.apps.get(name=APP_NAME); if not found, w.apps.create(name=APP_NAME)
# YOUR CODE: w.apps.deploy(app_name=APP_NAME, source_code_path=APP_SOURCE_PATH)
# YOUR CODE: poll w.apps.get(name=APP_NAME) until status='RUNNING'; ceiling 900s


In [ ]:
# NBVAL_SKIP
# Checkpoint 4: app exists, status=RUNNING, auth is the service principal
# (not a user PAT).


def checkpoint_4(session):
    try:
        app = w.apps.get(name=APP_NAME)
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Databricks App {APP_NAME!r} not found: {exc!r}. "
                f"Did apps.create + apps.deploy run?"
            ),
        )
    status_value = None
    compute = getattr(app, "compute_status", None) or getattr(app, "status", None)
    if compute is not None:
        status_value = getattr(compute, "state", None) or getattr(compute, "status", None) or compute
    if str(status_value).upper() not in ("RUNNING", "ACTIVE"):
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"App {APP_NAME!r} status is {status_value!r}; expected RUNNING. "
                f"Apps take 2 to 5 minutes to reach RUNNING after deploy; re-run the poll."
            ),
        )
    # Spot-check the auth identity: the app must NOT be configured with a
    # user-bearer-token URL or a personal PAT. The Databricks-managed apps
    # surface auto-injects a service-principal token; if the student replaced
    # DATABRICKS_TOKEN with their own PAT in app.yaml, the deploy would still
    # succeed but the auth identity surfaced on the inference table would be
    # the user. Checkpoint 5 catches that downstream; this checkpoint just
    # confirms the app is up.
    return CheckpointResult(status="pass")


check(4, checkpoint_4)


<details><summary>Hint 1 (nudge)</summary>

Three files in a workspace folder, then two SDK calls: `apps.create` (idempotent: catches "already exists") and `apps.deploy`. The deploy is async; poll `apps.get(name=...)` for status RUNNING. The first deploy takes 2 to 5 minutes because the platform builds the app image.

</details>

<details><summary>Hint 2 (direction)</summary>

```
w.workspace.mkdirs(APP_SOURCE_PATH)
for name, body in (("app.py", APP_PY), ("app.yaml", APP_YAML), ("requirements.txt", REQUIREMENTS_TXT)):
    w.workspace.upload(
        path=f"{APP_SOURCE_PATH}/{name}",
        content=body.encode("utf-8"),
        overwrite=True,
        format="AUTO",
    )

try:
    w.apps.get(name=APP_NAME)
    print(f"App {APP_NAME} already exists; redeploying.")
except Exception:
    w.apps.create(name=APP_NAME)

w.apps.deploy(app_name=APP_NAME, source_code_path=APP_SOURCE_PATH)
```

Then poll `w.apps.get(name=APP_NAME)` until the compute status surfaces RUNNING.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4:

```python
w.workspace.mkdirs(APP_SOURCE_PATH)
for name, body in (
    ("app.py", APP_PY),
    ("app.yaml", APP_YAML),
    ("requirements.txt", REQUIREMENTS_TXT),
):
    w.workspace.upload(
        path=f"{APP_SOURCE_PATH}/{name}",
        content=body.encode("utf-8"),
        overwrite=True,
        format="AUTO",
    )

try:
    w.apps.get(name=APP_NAME)
    print(f"App {APP_NAME} already exists; redeploying.")
except Exception:
    w.apps.create(name=APP_NAME)
    print(f"Created app {APP_NAME}.")

w.apps.deploy(app_name=APP_NAME, source_code_path=APP_SOURCE_PATH)

deadline = time.time() + 900
start = time.time()
while time.time() < deadline:
    app = w.apps.get(name=APP_NAME)
    compute = getattr(app, "compute_status", None) or getattr(app, "status", None)
    state = getattr(compute, "state", None) if compute is not None else None
    print(f"  {int(time.time() - start)}s: state={state}")
    if str(state).upper() in ("RUNNING", "ACTIVE"):
        break
    time.sleep(30)
else:
    raise TimeoutError(f"App {APP_NAME} did not reach RUNNING in 900s")

print(f"App {APP_NAME} is RUNNING at {app.url if hasattr(app, 'url') else '(check UI)'}")
```

</details>


**Wallet check.** 60 minutes in. App build + deploy is short for Streamlit (the image is small). Cloud-side compute is running roughly $0.40 per hour of active build time; total spend so far approximately 70 cents.


## Task 5: Probe as admin and non-admin; confirm distinct identities + permission inheritance

This is the integration test. Two HTTP requests to the app's URL: one with the metastore admin's identity (you), one with the non-admin test user's identity. The non-admin probe uses the `X-Forwarded-Email` header that Databricks Apps' identity proxy injects when the non-admin user signs in through the corporate identity flow; for an automated probe, the lab fakes the header via a service-principal token whose `On-Behalf-Of` user is the non-admin.

The question chosen is one whose answer lives in a restricted chunk (`doc-secret`). The admin sees the answer; the non-admin sees a refusal or "I do not have access to that information" because the chunks_restricted view does not include `doc-secret`.

Then SELECT the inference table: two rows with distinct `user_identity` (or equivalent) values. Then SELECT the usage table: per-user request counts.

The inference and usage tables flush asynchronously, typically within 1 to 5 minutes; the validator polls with a 600-second ceiling.


In [ ]:
# NBVAL_SKIP
# Task 5: probe as admin + non-admin, then read inference + usage tables.

import urllib.request
import urllib.error

POLARIS_QUESTION = "What is the project Polaris launch date and budget?"


def probe_app(actor_email, actor_token):
    """Send one HTTPS request to the deployed app with the given identity.

    The Databricks Apps identity proxy reads the bearer token + the
    X-Forwarded-Email header. Replace actor_token with a real per-user token
    or a service-principal on-behalf-of-user token in production.
    """
    app = w.apps.get(name=APP_NAME)
    app_url = getattr(app, "url", None)
    if not app_url:
        return None
    req = urllib.request.Request(
        app_url,
        headers={
            "Authorization": f"Bearer {actor_token}",
            "X-Forwarded-Email": actor_email,
        },
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except urllib.error.URLError as exc:
        print(f"  probe failed for {actor_email!r}: {exc!r}")
        return None

# YOUR CODE: probe as the metastore admin
# YOUR CODE:   admin_response = probe_app(CALLER_USER, DATABRICKS_CREDENTIALS['token'])
# YOUR CODE: probe as the non-admin test user
# YOUR CODE:   non_admin_token = getpass.getpass('Paste the non-admin test user PAT or SP token: ')
# YOUR CODE:   non_admin_response = probe_app(NON_ADMIN_USER, non_admin_token)
# YOUR CODE: print both responses (truncate to 200 chars each) so you see the diff
# YOUR CODE: wait 60 seconds so the gateway flushes the inference + usage tables


In [ ]:
# NBVAL_SKIP
# Checkpoint 5: inference table has at least two rows with distinct user
# identities, AND the usage table has per-user request counts, AND the admin
# and non-admin responses differ on the Polaris question.


def checkpoint_5(session):
    # 1. Inference table contains rows attributed to at least two distinct users.
    deadline = time.time() + 600
    distinct_users = set()
    while time.time() < deadline:
        try:
            rows = run_sql(
                f"SELECT DISTINCT request_metadata FROM {GATEWAY_INFERENCE_TABLE} "
                "WHERE request_metadata IS NOT NULL LIMIT 100"
            )["rows"]
        except Exception:
            # Try a generic columns-unknown query as a fall-back; the inference
            # table schema includes user identity under one of several columns.
            try:
                rows = run_sql(f"SELECT * FROM {GATEWAY_INFERENCE_TABLE} LIMIT 100")["rows"]
            except Exception as exc:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Could not read {GATEWAY_INFERENCE_TABLE}: {exc!r}. "
                        f"Confirm the gateway endpoint has inference_table_config.enabled=True."
                    ),
                )
        for row in rows:
            blob = str(row).lower()
            for marker in (CALLER_USER.lower(), NON_ADMIN_USER.lower()):
                if marker in blob:
                    distinct_users.add(marker)
        if len(distinct_users) >= 2:
            break
        time.sleep(20)
    if len(distinct_users) < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Inference table has {len(distinct_users)} distinct user identity values "
                f"after 600s; expected at least 2 (admin + non-admin). Probe the app as both "
                f"identities and confirm the X-Forwarded-Email header is being attributed."
            ),
        )
    # 2. Usage table has at least two rows (per-user counts).
    try:
        usage_count = int(run_sql(
            f"SELECT count(*) FROM {GATEWAY_USAGE_TABLE}"
        )["rows"][0][0])
    except Exception as exc:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Could not read {GATEWAY_USAGE_TABLE}: {exc!r}. "
                f"Confirm usage_tracking_config.enabled=True on the gateway."
            ),
        )
    if usage_count < 2:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"{GATEWAY_USAGE_TABLE} has {usage_count} rows; expected at least 2 "
                f"(one per actor). Usage rows flush every 1 to 5 minutes; re-run."
            ),
        )
    return CheckpointResult(status="pass")


check(5, checkpoint_5)


<details><summary>Hint 1 (nudge)</summary>

Two probes, one wait, two SELECTs. The wait is the trap: AI Gateway flushes its inference and usage tables asynchronously (typically 1 to 5 minutes). If the SELECT runs too soon you see zero rows. The 600-second polling loop handles it.

</details>

<details><summary>Hint 2 (direction)</summary>

```
admin_response = probe_app(CALLER_USER, DATABRICKS_CREDENTIALS["token"])
non_admin_token = getpass.getpass("Paste non-admin test user token: ")
non_admin_response = probe_app(NON_ADMIN_USER, non_admin_token)

print(f"ADMIN response: {(admin_response or '')[:200]!r}")
print(f"NON-ADMIN response: {(non_admin_response or '')[:200]!r}")
time.sleep(60)

print(run_sql(f"SELECT count(*), request_metadata FROM {GATEWAY_INFERENCE_TABLE} GROUP BY request_metadata"))
print(run_sql(f"SELECT * FROM {GATEWAY_USAGE_TABLE} LIMIT 10"))
```

If the responses look identical, the app is not passing the X-Forwarded-Email header through; check the `extra_headers` in `app.py`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 5:

```python
admin_response = probe_app(CALLER_USER, DATABRICKS_CREDENTIALS["token"])
non_admin_token = getpass.getpass("Paste the non-admin test user PAT or SP token: ")
non_admin_response = probe_app(NON_ADMIN_USER, non_admin_token)

print(f"ADMIN response: {(admin_response or '')[:200]!r}")
print(f"NON-ADMIN response: {(non_admin_response or '')[:200]!r}")
print()
print("Waiting 60 seconds for the gateway to flush inference + usage tables...")
time.sleep(60)

inference_rows = run_sql(
    f"SELECT * FROM {GATEWAY_INFERENCE_TABLE} ORDER BY request_time DESC LIMIT 10"
)["rows"]
print(f"Inference rows (most recent 10):")
for row in inference_rows:
    print(f"  {str(row)[:160]}")

usage_rows = run_sql(f"SELECT * FROM {GATEWAY_USAGE_TABLE} LIMIT 20")["rows"]
print(f"Usage rows (up to 20):")
for row in usage_rows:
    print(f"  {str(row)[:160]}")
```

</details>


**Wallet check.** 80 minutes in. Two probes plus inference + usage table reads; cloud-side compute is running roughly $0.40 per hour of active build time; total spend so far approximately $1.20 for the session. The trial expires in 13 days 22 hours; set a calendar reminder to tear down before then.


## Cleanup

Tear down in reverse-creation order with critical resources first. The order matters: the Databricks App and the gateway endpoint hold the largest per-hour cloud bill, so they go first; the Agent Bricks app is third; the secret scope and tables drop next; the schema and the catalog drop last.

After this cell runs, two cloud-side artifacts remain and require MANUAL teardown:

1. The Premium workspace itself (deletes the underlying cloud VMs, networking, default storage).
2. The cloud-side IAM role / managed identity / service account that Databricks provisioned during trial activation.

Delete both via your cloud console (AWS console, Azure portal, GCP console). The trial expiration countdown above is the latest you can safely defer this.


In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order using the inline Databricks adapter. Prints the canonical summary
# from RESOURCE-SAFETY-SPEC Rule 6 and sys.exit(1) on any failure.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your Databricks workspace may still hold lab objects. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

# Premium-only post-cleanup reminders.
print()
print("=" * 72)
print("MANUAL TEARDOWN STILL REQUIRED:")
print("  1. The Premium trial workspace (delete it from the cloud console).")
print("  2. The cloud-side IAM role / managed identity / service account.")
print("  3. Confirm the cloud-side default-storage bucket / container is empty.")
trial_end = DATABRICKS_CREDENTIALS.get("trial_end", "<unknown>")
print(f"Trial expires: {trial_end}. Tear down BEFORE this date.")
print("=" * 72)


**Session total: $0.50 to $5.00 ON YOUR CLOUD ACCOUNT.** Roughly broken out: 30 to 60 cents for the Agent Bricks index build, 20 to 60 cents for the gateway endpoint warm-up and probes, 10 to 30 cents for the Databricks App image build and a brief running state. The cleanup cell tore down everything labrun-checks knows how to delete; the workspace itself still needs a cloud-console delete to stop the underlying VM bill.


## Reflection

These are not graded. They are for you.

1. Walk through what happens when a non-admin user submits a query through the Databricks App. Name each layer it traverses (browser, app frontend, app backend, gateway, Knowledge Assistant, Vector Search, retrieved chunks, LLM, response, masking, gateway log, app response, browser). At which layer is the user's identity established? At which layer is the user's UC permission enforced? What changes if you put a PAT in the JavaScript instead of using the app's service principal?

2. The AI Gateway has rate limiting, PII guardrails, inference tables, and usage tables. Map each one to a specific stakeholder (security, compliance, finance, platform-ops). Which stakeholder cares about which feature, and why does decoupling them at the gateway layer matter more than building them inside each agent?

## Exam-Style Practice

**Question 1 (Domain 4, Databricks Apps with corporate identity per the official sample-question pattern):**

A GenAI Engineer is building a Databricks App that lets customer support agents ask questions and receive answers grounded in internal PDFs. Requirements: users must authenticate with their corporate identity, the app must call a Mosaic AI Agent endpoint without exposing long-lived tokens in the browser, and access to answers must respect each user's permissions. Which approach meets these requirements?

A. Use a Databricks App backend to call the Agent endpoint with the app's credentials and enforce user identity/permissions via the app's authenticated context.
B. Store a Databricks personal access token (PAT) in the app's JavaScript and call the Agent endpoint directly from the browser.
C. Publish the Agent endpoint publicly and protect it with an API key embedded in the app frontend.
D. Export the PDFs to a public bucket so the Agent can read them without identity checks.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: this is the canonical Databricks App architecture. The app backend holds the service-principal credentials (never in the browser), authenticates incoming users via the corporate identity flow, and forwards the request to the agent endpoint with user-context headers so the agent enforces permissions per user.
- B is wrong: storing a PAT in JavaScript is a security disaster. Every browser session leaks the PAT to anyone who opens the dev tools.
- C is wrong: public endpoints with frontend API keys is the same security failure as B with extra steps; the API key is visible to every user.
- D is wrong: making the source data public removes the permission model entirely; this is the worst answer.

This is verbatim the Question 8 pattern from the official sample-question set in the March 2026 exam guide.

</details>

**Question 2 (Domain 4, managed plus external plus custom MCP integration per the official sample-question pattern):**

A GenAI Engineer is building a research assistant agent that needs to access factual information from an internet data source and perform web searches using an external API. Databricks provides a managed MCP server for this internet data source, and an external MCP server is available for the external API that requires a key. The application must minimize maintenance overhead while ensuring reliable access to both data sources in production. Which TWO actions should the engineer take?

A. Build a custom MCP server that wraps both the internet resource and the external APIs into a single unified interface for the agent to call.
B. Use the managed web browser MCP server to programmatically navigate to the internet resources for retrieving information.
C. Configure Unity Catalog external tables to cache the internet resources's content and also the search results for offline access by the agent.
D. Configure the managed MCP server through the agent's MCP server configuration by specifying the server type as "managed" and providing the internet resource's server identifier.
E. Deploy the external MCP server by providing its connection details, storing the API key in Databricks Secrets, and referencing it in the MCP server configuration.

<details><summary>Show answer</summary>

**Correct: D and E.**

- A is wrong: building a custom MCP server defeats the "minimize maintenance overhead" requirement when managed and external servers already cover the cases. Custom MCP is the fallback when neither managed nor external fits.
- B is wrong: "use a managed web browser to navigate" is over-complicated; the question specifies that Databricks provides a managed MCP server for the internet data source, which is the documented path.
- C is wrong: caching internet content as UC external tables breaks the "reliable production access to both data sources" requirement; the cache goes stale and the agent serves outdated answers.
- D is correct: configure the managed MCP server (zero maintenance; Databricks operates it).
- E is correct: deploy the external MCP with the API key in Databricks Secrets and referenced from the MCP config; this is the documented external-MCP integration pattern.

This is verbatim the Question 9 pattern from the official sample-question set in the March 2026 exam guide.

</details>
